# C-009: Cross-Regional Harassment Network Using Shared Infrastructure

**Synthetic investigation portfolio.** All users, targets, devices, content, and events are fictional. Automated outputs are investigative leads requiring human validation.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

BASE = Path('..')
DB_PATH = BASE / 'database' / 'openai_abuse_investigator_synthetic.sqlite'
CASE_ID = 'C-009'
conn = sqlite3.connect(DB_PATH)


## 1. Case and Event Overview


In [ ]:
case = pd.read_sql_query("SELECT * FROM cases WHERE case_id='C-009'", conn)
events = pd.read_sql_query("SELECT * FROM events WHERE case_id='C-009'", conn)
events['timestamp'] = pd.to_datetime(events['timestamp'], utc=True)
events['raw_risk_score'] = pd.to_numeric(events['raw_risk_score'])
display(case)
events[['event_type','raw_risk_score']].describe(include='all')


## 2. Cross-Case Accounts


In [ ]:
cross_case_accounts = pd.read_sql_query('''
SELECT ca.account_id, GROUP_CONCAT(DISTINCT ca.case_id) AS cases,
       COUNT(DISTINCT ca.case_id) AS case_count, a.role,
       a.prior_enforcement, a.risk_tier
FROM case_accounts ca
JOIN accounts a ON a.account_id=ca.account_id
WHERE ca.account_id IN (SELECT account_id FROM case_accounts WHERE case_id='C-009')
GROUP BY ca.account_id, a.role, a.prior_enforcement, a.risk_tier
HAVING COUNT(DISTINCT ca.case_id)>1
ORDER BY case_count DESC, ca.account_id
''', conn)
cross_case_accounts


## 3. Shared Infrastructure


In [ ]:
shared_devices = pd.read_sql_query('''
SELECT s.device_id, COUNT(DISTINCT e.case_id) AS case_count,
       GROUP_CONCAT(DISTINCT e.case_id) AS cases,
       COUNT(DISTINCT s.account_id) AS account_count
FROM sessions s JOIN events e ON e.session_id=s.session_id
WHERE s.device_id IN (
 SELECT DISTINCT s2.device_id FROM sessions s2
 JOIN case_accounts ca ON ca.account_id=s2.account_id WHERE ca.case_id='C-009'
)
GROUP BY s.device_id
HAVING COUNT(DISTINCT e.case_id)>1
ORDER BY case_count DESC, account_count DESC
''', conn)
shared_devices


## 4. Role and Risk Analysis


In [ ]:
role_summary = pd.read_sql_query('''
SELECT a.role, COUNT(*) AS event_count,
       ROUND(AVG(CAST(e.raw_risk_score AS REAL)),3) AS avg_risk,
       COUNT(DISTINCT e.account_id) AS account_count
FROM events e JOIN accounts a ON a.account_id=e.account_id
WHERE e.case_id='C-009'
GROUP BY a.role ORDER BY event_count DESC
''', conn)
display(role_summary)
role_summary.sort_values('event_count').plot.barh(x='role', y='event_count', legend=False)
plt.title('C-009 Event Volume by Role')
plt.xlabel('Events')
plt.show()


## 5. Analyst Assessment

The working hypothesis is that C-009 represents a coordinated network rather than coincidental parallel activity. The conclusion must be tested against shared infrastructure, cross-case accounts, repeated content families, role differentiation, post-enforcement behavior, and alternative benign explanations.
